                # Sweeps And Dispersive

                This notebook documents the sweep APIs for both single systems and coupled systems, then uses lookup-aware post-processing to extract dispersive information.
                

                **Audience:** readers who want to study parameter-dependent spectra and dispersive trends.

                **Prerequisites:** the `HilbertSpace` and lookup workflow from notebook 05.

                **Learning goals:** use `SingleSystemSweep`, construct a `ParameterSweep`, run it explicitly, build lookup-aware summaries, and use the Makie sweep-plotting helpers.
                

                ## Outline

                1. Sweep a single `TunableTransmon` with `SingleSystemSweep`.
                2. Build a coupled-system `ParameterSweep` with explicit `run!`.
                3. Query dressed energies and bare/dressed labels at individual parameter points.
                4. Compute `chi_matrix`, `self_kerr`, and `lamb_shift`, and plot sweep summaries.
                

In [ ]:
                using ScQubitsMimic
                using CairoMakie
                

In [ ]:
                tmon = TunableTransmon(EJmax=20.0, EC=0.3, d=0.1, flux=0.0, ng=0.0, ncut=15, truncated_dim=4)
                single_sweep = SingleSystemSweep(tmon, :flux, collect(range(0.0, 0.5; length=7)); evals_count=4)

                (
                    param_name = single_sweep.param_name,
                    param_vals = single_sweep.param_vals,
                    ω01_vs_flux = round.(single_sweep.spectrum.eigenvalues[:, 2] .- single_sweep.spectrum.eigenvalues[:, 1], digits=6),
                )
                

In [ ]:
                plot_evals_vs_paramvals(single_sweep; subtract_ground=true, evals_count=4)
                

In [ ]:
                coupled_tmon = TunableTransmon(EJmax=20.0, EC=0.3, d=0.1, flux=0.0, ng=0.0, ncut=10, truncated_dim=5)
                resonator = Oscillator(E_osc=5.5, truncated_dim=8)
                hs = HilbertSpace([coupled_tmon, resonator])
                add_interaction!(hs, 0.05, [coupled_tmon, resonator],
                    [s -> n_operator(s), s -> annihilation_operator(s) + creation_operator(s)])

                sweep = ParameterSweep(
                    hs,
                    Dict(:flux => collect(range(0.0, 0.3; length=4))),
                    (hs, vals) -> begin
                        hs.subsystems[1].flux = vals[:flux]
                    end;
                    evals_count=20,
                    store_lookups=true,
                    ignore_low_overlap=true,
                    autorun=false,
                )

                run!(sweep)
                (
                    param_order = sweep.param_order,
                    dressed_shape = size(sweep.dressed_evals),
                    lookup_built = lookup_exists(sweep),
                )
                

In [ ]:
                (
                    point_2_bare_of_dressed_3 = bare_index(sweep, 3; param_indices=(2,)),
                    point_2_dressed_of_bare_2_1 = dressed_index(sweep, (2, 1); param_indices=(2,)),
                    point_2_energy_of_dressed_3 = round(energy_by_dressed_index(sweep, 3; param_indices=(2,)), digits=6),
                    point_2_energy_of_bare_2_1 = round(energy_by_bare_index(sweep, (2, 1); param_indices=(2,)), digits=6),
                )
                

In [ ]:
                chi = chi_matrix(sweep)
                (
                    chi_shape = size(chi),
                    chi_12_MHz = round.(chi[:, 1, 2] .* 1000, digits=3),
                    self_kerr_q1_MHz = round.(self_kerr(sweep, 1) .* 1000, digits=3),
                    lamb_shift_q1_MHz = round.(lamb_shift(sweep, 1) .* 1000, digits=3),
                )
                

In [ ]:
                plot_chi_vs_paramvals(sweep; subsys_pair=(1, 2))
                

                ## Exercise

                Increase the coupling from `0.05` to `0.08` GHz, rerun the sweep, and compare the new `χ12` values against the original sweep.
                

In [ ]:
                hs_exercise = HilbertSpace([deepcopy(coupled_tmon), Oscillator(E_osc=5.5, truncated_dim=8)])
                add_interaction!(hs_exercise, 0.08, hs_exercise.subsystems,
                    [s -> n_operator(s), s -> annihilation_operator(s) + creation_operator(s)])

                sweep_exercise = ParameterSweep(
                    hs_exercise,
                    Dict(:flux => collect(range(0.0, 0.3; length=4))),
                    (hs, vals) -> begin
                        hs.subsystems[1].flux = vals[:flux]
                    end;
                    evals_count=20,
                    store_lookups=true,
                    ignore_low_overlap=true,
                )

                round.(chi_matrix(sweep_exercise)[:, 1, 2] .* 1000, digits=3)
                

                ## Pitfalls And Extensions

                `plot_chi_vs_paramvals` depends on `chi_matrix(sweep)`, which in turn needs lookup data rich enough to resolve the first two excitations of each subsystem. If the plot fails, increase `evals_count` and rebuild the sweep.

                `ParameterSweep` defaults to `autorun=true`. Using `autorun=false` plus `run!` is often clearer in documentation because it makes the execution boundary explicit.
                

                ## API Covered

                `SingleSystemSweep`, `ParameterSweep`, `run!`, `plot_evals_vs_paramvals`, `chi_matrix`, `self_kerr`, `lamb_shift`, and `plot_chi_vs_paramvals`.
                